# Modeling the domain before the tables

Core stage, lesson C16. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** name the entities, attributes and relationships of a business, read an entity-relationship diagram, turn a many-to-many link into a table of its own, declare keys, and say what one row of each table stands for before you write CREATE TABLE.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import decimal
import math
import numbers
import subprocess
import sys

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb

# Run this cell as it is. It also holds the answer checker and the helpers of the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(got, decimal.Decimal):
            got = int(got) if got == got.to_integral_value() else float(got)
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong


def show(sql, limit=12):
    """Run a statement and print its rows as text: the column names, then one line for each row."""
    cur = con.execute(sql)
    if cur.description is None:
        print("(the statement ran and returns no rows)")
        return
    rows = cur.fetchall()
    print(" | ".join(column[0] for column in cur.description))
    for row in rows[:limit]:
        print(" | ".join("NULL" if value is None else str(value) for value in row))
    if len(rows) > limit:
        print(f"... {len(rows)} rows in all")


def rows(sql):
    """Run a query and return its rows as lists, with decimals as numbers and dates as text."""
    cur = con.execute(sql)
    return [[float(v) if isinstance(v, decimal.Decimal) else (str(v) if hasattr(v, "isoformat") else v) for v in row] for row in cur.fetchall()]


def say_error(sql):
    """Run a statement; when the database refuses it, print the first line of its message instead of stopping."""
    try:
        show(sql)
    except duckdb.Error as error:
        print("Refused:", str(error).splitlines()[0])


def key_listing(table):
    """The primary and foreign keys of a table, as [type, column, position] lists (the query of unit 3 of C16)."""
    return [list(r) for r in con.execute(KEYS, [table]).fetchall()]


KEYS = """SELECT tc.constraint_type, kcu.column_name, kcu.ordinal_position AS position
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
  ON kcu.constraint_name = tc.constraint_name AND kcu.table_name = tc.table_name
WHERE tc.table_name = ? AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY tc.constraint_type, kcu.ordinal_position, kcu.column_name"""

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/cafe-sales/v1/"
LOCAL = "../../datasets/cafe-sales/v1/"      # the same files, when this notebook runs inside a clone of the repository
TABLES = {                                   # the columns and types of datafromzero's cafe-sales.sql (no keys or constraints there)
    "stores": "'INTEGER','VARCHAR','VARCHAR','DATE'",
    "customers": "'INTEGER','DATE','VARCHAR','VARCHAR','INTEGER'",
    "products": "'INTEGER','VARCHAR','VARCHAR','DECIMAL(6,2)'",
    "orders": "'INTEGER','INTEGER','INTEGER','TIMESTAMP','VARCHAR','VARCHAR'",
    "order_items": "'INTEGER','INTEGER','INTEGER','INTEGER','DECIMAL(6,2)'",
}


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    import os
    import tempfile
    import urllib.request
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"


con = duckdb.connect()                       # a new, empty database every time this cell runs
for name, types in TABLES.items():
    con.execute(f"CREATE TABLE {name} AS FROM read_csv('{fetch(name)}', types=[{types}])")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
counts = [con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] for name in TABLES]
assert counts == [3, 150, 16, 1800, 3287]
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: Entities and attributes

**Predict first.** the owner names these nouns: store, customer, order, line, product, payment method, category, city and loyalty tier. Which would you give a table of its own?

In [ ]:
show("""SELECT table_name,
       COUNT(*) AS columns,
       string_agg(column_name, ', ' ORDER BY ordinal_position) AS column_names
FROM information_schema.columns
WHERE table_schema = 'main'
GROUP BY table_name
ORDER BY table_name""")

## Unit 2: Relationships and cardinality

**Predict first.** one customer can have many orders. What is the fewest orders a customer has, and can an order have no customer?

In [ ]:
show("""SELECT COUNT(*) AS customers,
       MIN(n) AS fewest_orders,
       MAX(n) AS most_orders,
       COUNT(*) FILTER (WHERE n = 0) AS never_ordered,
       (SELECT COUNT(*) FROM orders WHERE customer_id IS NULL) AS orders_without_customer
FROM (SELECT c.customer_id, COUNT(o.order_id) AS n
      FROM customers AS c
      LEFT JOIN orders AS o ON o.customer_id = c.customer_id
      GROUP BY c.customer_id)""")

## Unit 3: A link table for many to many

**Predict first.** will the number of pairs equal the number of lines? If it does, which two columns could be the primary key of the table?

In [ ]:
show("""SELECT COUNT(*) AS lines,
       COUNT(DISTINCT order_id) AS orders,
       COUNT(DISTINCT product_id) AS products,
       COUNT(DISTINCT (order_id, product_id)) AS pairs
FROM order_items""")

The tables above were loaded from files and declare no keys. A foreign key needs a key to point at, so this cell declares two first, then creates the link table `store_menu` and puts in one pair. (Run all from the top starts from a new database; running this one cell twice would try to declare the keys twice.)

In [ ]:
con.execute("""ALTER TABLE stores ADD PRIMARY KEY (store_id);
ALTER TABLE products ADD PRIMARY KEY (product_id);

DROP TABLE IF EXISTS store_menu;
CREATE TABLE store_menu (
  store_id   INTEGER NOT NULL REFERENCES stores (store_id),
  product_id INTEGER NOT NULL REFERENCES products (product_id),
  listed_on  DATE,
  PRIMARY KEY (store_id, product_id)
);

INSERT INTO store_menu VALUES (1, 3, DATE '2026-01-01')""")
print("One pair is in.")

**Predict first.** the next statement repeats the first pair with another date. What happens?

In [ ]:
say_error("INSERT INTO store_menu VALUES (1, 3, DATE '2026-02-01')")

Ask the database what you declared:

In [ ]:
for line in key_listing("store_menu"):
    print(line)

## Unit 4: The grain, and what a wrong one costs

**Predict first.** the total of each order, one row for each order, added up: how many rows and what total?

In [ ]:
ORDER_TOTALS = """WITH order_totals AS (
  SELECT i.order_id, SUM(i.quantity * p.unit_price_usd - i.discount_usd) AS order_total
  FROM order_items AS i
  JOIN products AS p ON p.product_id = i.product_id
  GROUP BY i.order_id
)
"""
show(ORDER_TOTALS + "SELECT COUNT(*) AS rows, ROUND(SUM(order_total), 2) AS total FROM order_totals")

**Predict first.** every line has its order, so the join is correct. Will the total stay the same, grow a little, or grow a lot?

In [ ]:
show(ORDER_TOTALS + """SELECT COUNT(*) AS rows, ROUND(SUM(t.order_total), 2) AS total
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
JOIN order_totals AS t ON t.order_id = o.order_id""")

## Finish it

### Exercise 1: Finish it

**Task:** Link the guests to the tastings. A guest can sign up for many tastings, and a tasting has many guests. Each signup records how many seats the guest took. The two entity tables are given, with `rsvp` half built: finish it, so that it has one foreign key for each side and a primary key that allows one signup for each guest and tasting.

**Expected:** `answer_1` is the list of constraints of `rsvp`: 4 lists of [type, column, position].

In [ ]:
con.execute("""DROP TABLE IF EXISTS rsvp;
DROP TABLE IF EXISTS guest;
DROP TABLE IF EXISTS tasting;

CREATE TABLE guest (guest_id INTEGER PRIMARY KEY, guest_name VARCHAR NOT NULL);
CREATE TABLE tasting (tasting_id INTEGER PRIMARY KEY, held_on DATE NOT NULL);

-- grain of rsvp: one row for each guest and tasting
CREATE TABLE rsvp (
  guest_id INTEGER NOT NULL REFERENCES guest (guest_id),
  seats    INTEGER NOT NULL CHECK (seats > 0)
)""")
answer_1 = key_listing("rsvp")

In [ ]:
_ = check("Exercise 1", "answer_1",
          [["FOREIGN KEY", "guest_id", 1], ["FOREIGN KEY", "tasting_id", 1], ["PRIMARY KEY", "guest_id", 1], ["PRIMARY KEY", "tasting_id", 2]],
          traps=[([["FOREIGN KEY", "guest_id", 1], ["FOREIGN KEY", "tasting_id", 1], ["PRIMARY KEY", "guest_id", 1]], "A primary key on guest_id alone lets each guest sign up for one tasting only. The pair of keys identifies a signup."),
                 ([["FOREIGN KEY", "guest_id", 1]], "Only one side is linked so far. The link table needs a foreign key for each side and a primary key over the pair.")])

<details>
<summary>Hint</summary>

A link table has one column with `REFERENCES` for each side, and a `PRIMARY KEY` over the pair. Here the second side is `tasting`.

</details>

<details>
<summary>Half-solution</summary>

```python
CREATE TABLE rsvp (
  guest_id   INTEGER NOT NULL REFERENCES guest (guest_id),
  tasting_id ...,
  seats      INTEGER NOT NULL CHECK (seats > 0),
  PRIMARY KEY (...)
)
```

</details>

<details>
<summary>Solution</summary>

```python
con.execute("""DROP TABLE IF EXISTS rsvp;
DROP TABLE IF EXISTS guest;
DROP TABLE IF EXISTS tasting;

CREATE TABLE guest (guest_id INTEGER PRIMARY KEY, guest_name VARCHAR NOT NULL);
CREATE TABLE tasting (tasting_id INTEGER PRIMARY KEY, held_on DATE NOT NULL);

-- grain of rsvp: one row for each guest and tasting
CREATE TABLE rsvp (
  guest_id   INTEGER NOT NULL REFERENCES guest (guest_id),
  tasting_id INTEGER NOT NULL REFERENCES tasting (tasting_id),
  seats      INTEGER NOT NULL CHECK (seats > 0),
  PRIMARY KEY (guest_id, tasting_id)
)""")
answer_1 = key_listing("rsvp")
```

The listing reads FOREIGN KEY `guest_id` and `tasting_id`, then PRIMARY KEY `guest_id` (position 1) and `tasting_id` (position 2). `seats` stays outside the key: it describes the pair. A primary key on `guest_id` alone would let each guest sign up for one tasting only, which the check cell reports.

</details>

## On your own

### Exercise 2: On your own

**Task:** Track the baristas. A barista has a name and works at one cafe, and a cafe has many baristas. A barista can earn many certificates (such as latte art), and a certificate is held by many baristas, each with the day it was earned. Before you start, say what the result looks like: three tables, and for each a listing of its constraints. Write `barista` (with `barista_id` and its cafe, `store_id`), `certificate` (with `certificate_id`) and the link table `barista_certificate`, each with a comment that states its grain. The cafe is `stores`, which needs the keys of unit 3. Start the cell with `DROP TABLE IF EXISTS` lines, children before parents, so that you can run it again.

**Expected:** `answer_2` is a dictionary with the constraint listing of `barista` and of `barista_certificate`.

In [ ]:
# Exercise 2: write your CREATE TABLE statements here, inside con.execute(""" ... """), then run the check cell.

In [ ]:
answer_2 = {table: key_listing(table) for table in ("barista", "barista_certificate")}      # the constraints of your two tables
_ = check("Exercise 2", "answer_2",
          {"barista": [["FOREIGN KEY", "store_id", 1], ["PRIMARY KEY", "barista_id", 1]],
           "barista_certificate": [["FOREIGN KEY", "barista_id", 1], ["FOREIGN KEY", "certificate_id", 1], ["PRIMARY KEY", "barista_id", 1], ["PRIMARY KEY", "certificate_id", 2]]},
          traps=[({"barista": [["FOREIGN KEY", "store_id", 1], ["PRIMARY KEY", "barista_id", 1]],
                   "barista_certificate": [["FOREIGN KEY", "barista_id", 1], ["FOREIGN KEY", "certificate_id", 1], ["PRIMARY KEY", "barista_id", 1]]},
                  "A primary key on barista_id alone lets a barista hold one certificate only. The pair identifies a row."),
                 ({"barista": [["PRIMARY KEY", "barista_id", 1]],
                   "barista_certificate": [["FOREIGN KEY", "barista_id", 1], ["FOREIGN KEY", "certificate_id", 1], ["PRIMARY KEY", "barista_id", 1], ["PRIMARY KEY", "certificate_id", 2]]},
                  "A barista works at one cafe: a column of barista holds the key of stores, and it needs REFERENCES.")])

<details>
<summary>Hint</summary>

Two relationships, two shapes. The cafe of a barista is one-to-many, so a column of `barista` holds the key of `stores`. Certificates are many-to-many, so they need a table of their own.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A barista table with a primary key and one foreign key, and a link table with two foreign keys and a primary key over the pair."*

```python
CREATE TABLE barista (barista_id INTEGER PRIMARY KEY, barista_name VARCHAR NOT NULL, store_id INTEGER NOT NULL REFERENCES ...)
```

</details>

<details>
<summary>Solution</summary>

```python
con.execute("""DROP TABLE IF EXISTS barista_certificate;
DROP TABLE IF EXISTS certificate;
DROP TABLE IF EXISTS barista;

-- grain: one row for each barista
CREATE TABLE barista (
  barista_id   INTEGER PRIMARY KEY,
  barista_name VARCHAR NOT NULL,
  store_id     INTEGER NOT NULL REFERENCES stores (store_id)
);
-- grain: one row for each certificate
CREATE TABLE certificate (
  certificate_id INTEGER PRIMARY KEY,
  title          VARCHAR NOT NULL
);
-- grain: one row for each barista and certificate
CREATE TABLE barista_certificate (
  barista_id     INTEGER NOT NULL REFERENCES barista (barista_id),
  certificate_id INTEGER NOT NULL REFERENCES certificate (certificate_id),
  earned_on      DATE NOT NULL,
  PRIMARY KEY (barista_id, certificate_id)
)""")
```

The cafe is one-to-many, so it is a column of `barista`; certificates are many-to-many, so they get a link table, and `earned_on` sits there because it belongs to the pair. A barista who works at two cafes would turn the first relationship into a second link table: a rule of the owner's description decides.

</details>

## Check yourself

Without looking back: say the grain of each of the five cafe tables in one sentence. Then try the exercise.

<details>
<summary>answer</summary>

`stores`: one row for each cafe. `customers`: one row for each loyalty customer. `products`: one row for each menu item. `orders`: one row for each visit to the till. `order_items`: one row for each product on an order.

</details>

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague counts the customers who were referred by a gold customer and gets 1. Fix the join so that `answer_3` is the number of customers who were referred by a gold customer.

**Expected:** `answer_3` is a whole number.

In [ ]:
answer_3 = rows("""SELECT COUNT(*) AS referred_by_gold
FROM customers AS c
JOIN customers AS r ON c.customer_id = r.referred_by
WHERE r.loyalty_tier = 'gold'""")[0][0]      # colleague: "referred by a gold customer"

In [ ]:
_ = check("Exercise 3", "answer_3", 4, traps={1: "That counts the gold customers who were themselves referred by someone: the key sits on the customer who was referred, and holds the referrer. Which side of the ON is which?"})

<details>
<summary>Hint</summary>

Which table row holds `referred_by`: the person who referred, or the person who was referred? Say what `c` and `r` are in the query.

</details>

<details>
<summary>Half-solution</summary>

```python
JOIN customers AS r ON c.referred_by = ...
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = rows("""SELECT COUNT(*) AS referred_by_gold
FROM customers AS c
JOIN customers AS r ON c.referred_by = r.customer_id
WHERE r.loyalty_tier = 'gold'""")[0][0]
```

`referred_by` sits on the customer who was referred and holds the key of the referrer. The colleague matched `c.customer_id` to `r.referred_by`, so `r` is the referred customer and `c` the referrer: the query counts the gold customers who were referred by someone, 1. With the key matched the right way round, 4 customers were referred by a gold customer. A foreign key points from the many side to the one side: read it from the row that holds it.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")